In [1]:
# -*- coding: utf-8 -*-
"""FutureEnginner - Block Training.ipynb"""

import os

# ── Step 1: Extract dataset (manually upload dataset.zip first) ──
import zipfile
with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
    z.extractall('/content')

print("Dataset extracted!")
!find /content/dataset -type d

# ── Step 2: Verify labels ──
print(f"Train images: {len(os.listdir('/content/dataset/images/train'))}")
print(f"Val images:   {len(os.listdir('/content/dataset/images/val'))}")
print(f"Train labels: {len(os.listdir('/content/dataset/labels/train'))}")
print(f"Val labels:   {len(os.listdir('/content/dataset/labels/val'))}")

sample = os.listdir('/content/dataset/labels/train')[0]
with open(f'/content/dataset/labels/train/{sample}') as f:
    print(f"\nSample label ({sample}):")
    print(f.read())

# ── Step 3: Install ultralytics ──
!pip install ultralytics -q

# ── Step 4: Train ──
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

model.train(
    data='/content/dataset/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='wro_detector',
    patience=20,
    device=0
)

# ── Step 5: Validate ──
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# ── Step 6: Export to ONNX ──
model = YOLO('/content/runs/detect/wro_detector/weights/best.pt')
model.export(format='onnx', imgsz=640)

# ── Step 7: Download ──
from google.colab import files
files.download('/content/runs/detect/wro_detector/weights/best.onnx')

with open('/content/labelmap.txt', 'w') as f:
    f.write('green\nred\n')
files.download('/content/labelmap.txt')

print("\nDone! Files downloaded: best.onnx, labelmap.txt")

Dataset extracted!
/content/dataset
/content/dataset/images
/content/dataset/images/val
/content/dataset/images/train
/content/dataset/labels
/content/dataset/labels/val
/content/dataset/labels/train
Train images: 308
Val images:   77
Train labels: 308
Val labels:   77

Sample label (red_0133.txt):
1 0.710938 0.508333 0.262500 0.575000
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 4.9s, saved as '/content/runs/detect/wro_detector/weights/best.onnx' (9.4 MB)

Export complete (5.3s)
Results saved to /content/runs/detect/wro_detector/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/wro_detector/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/wro_detector/weights/best.onnx imgsz=640 data=/content/dataset/dataset.yaml  
Visualize:       https://netron.app


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Files downloaded: best.onnx, labelmap.txt
